# Colab v3 — Zephyr-7B SLA benchmark

**Runtime → A100 GPU.** Upload `InferenceLab_v3.zip`, run cells 1–9 in order.


In [ ]:
!nvidia-smi


In [ ]:
import os
os.chdir('/content')
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))


In [ ]:
import os, shutil, zipfile, glob

os.chdir('/content')
PROJECT_ROOT = '/content/inference_lab'
shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
os.makedirs(PROJECT_ROOT)

zip_name = 'InferenceLab_v3.zip'
if not os.path.exists(zip_name):
    zips = sorted(glob.glob('/content/*.zip'), key=os.path.getmtime)
    assert zips, 'Upload InferenceLab_v3.zip first'
    zip_name = zips[-1]

with zipfile.ZipFile(zip_name) as z:
    z.extractall(PROJECT_ROOT)

assert os.path.isdir(f'{PROJECT_ROOT}/server')
os.chdir(PROJECT_ROOT)
print('Working dir:', os.getcwd())


In [ ]:
import os, subprocess, sys
os.chdir('/content/inference_lab')
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'

!pip install -q -r requirements.txt
!pip uninstall -y vllm transformers tokenizers 2>/dev/null
!pip install -q "vllm==0.8.5" "transformers==4.51.3" "tokenizers<0.22"

env = os.environ.copy()
env['PYTHONPATH'] = '/content/inference_lab'
check = subprocess.run(
    [sys.executable, '-c',
     'from vllm import AsyncLLMEngine; import google.protobuf; '
     'print("OK protobuf", google.protobuf.__version__)'],
    capture_output=True, text=True, env=env, cwd='/content/inference_lab',
)
print(check.stdout.strip() or check.stderr.strip())
assert check.returncode == 0


In [ ]:
import json, subprocess, time, sys, os, urllib.request

PROJECT_ROOT = '/content/inference_lab'
os.chdir(PROJECT_ROOT)

def _tail(path, n=60):
    with open(path) as f:
        return ''.join(f.readlines()[-n:])

def wait_for_server(log_path, server, timeout=1800):
    start = time.time()
    while time.time() - start < timeout:
        if server.poll() is not None:
            print(_tail(log_path))
            return False
        try:
            with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5) as r:
                if json.loads(r.read()).get('status') == 'ok':
                    return True
        except Exception:
            pass
        time.sleep(3)
    print(_tail(log_path))
    return False

def run_strategy(app_module, strategy, runs=2, capstone=False, extra_env=None):
    env = os.environ.copy()
    env['PYTHONPATH'] = PROJECT_ROOT
    env.setdefault('USE_TF', '0')
    if extra_env:
        env.update(extra_env)
    log_path = f'/content/server_{strategy}.log'
    logf = open(log_path, 'w')
    server = subprocess.Popen(
        ['uvicorn', app_module, '--host', '127.0.0.1', '--port', '8000'],
        env=env, stdout=logf, stderr=subprocess.STDOUT, cwd=PROJECT_ROOT,
    )
    try:
        assert wait_for_server(log_path, server), f'{strategy} server failed'
        cmd = [sys.executable, 'scripts/run_benchmark_suite.py',
               '--url', 'http://127.0.0.1:8000', '--strategy', strategy,
               '--runs', str(runs), '--monitor-gpu']
        if capstone:
            cmd.append('--capstone')
        assert subprocess.run(cmd, env=env, cwd=PROJECT_ROOT).returncode == 0
    finally:
        server.terminate()
        try:
            server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            server.kill()
        logf.close()
        time.sleep(5)

print('Ready:', PROJECT_ROOT)


In [ ]:
SLA_ENV = {
    'VLLM_MODEL': 'zephyr-7b',
    'SLA_P95_BUDGET_SEC': '3.0',
    'SLA_MAX_QUEUE': '32',
}
run_strategy('server.sla_server:app', 'sla', runs=2, capstone=True, extra_env=SLA_ENV)
!ls -la results/load_sla_*.json


## Charts and download

In [ ]:
import os
os.chdir('/content/inference_lab')
os.makedirs('report', exist_ok=True)
!python scripts/plot_tail_latency.py --results-dir results --out-dir report --tier v3 --strategies vllm,sla
!python scripts/generate_tier_charts.py


In [ ]:
import os
os.chdir('/content/inference_lab')
!zip -r results_and_report.zip results report
from google.colab import files
files.download('results_and_report.zip')
